# Prepare Static Dataset

Dans ce notebook, le jeu de données utilisé pour les modèles de survie développés dans ce projet (Weibull AFT, Log-normal AFT, Log-logistic AFT, Random Survival Forest et XGBoost AFT) est préparé.

Une seule mesure de DFGe est conservée par patient (mesure de référence), à partir de laquelle sont construites les variables de survie et les variables cliniques utilisées pour la prédiction.

À l'issue de ce notebook, les jeux de données d'entraînement, de test interne et de test externe sont créés et sauvegardés afin d'être utilisés dans les différents notebooks de modélisation.

In [ ]:
import numpy as np
import pandas as pd
import polars as pl

from pathlib import Path
from scipy import stats
from sklearn.model_selection import train_test_split

### Les datasets

In [ ]:
# ==========================================================
# CHARGEMENT DES DONNÉES
# ==========================================================
inclusion = pl.read_parquet("Preprocessed_data/inclusion.parquet")
image = pl.read_parquet("Preprocessed_data/image.parquet")
creat = pl.read_parquet("Preprocessed_data/creat.parquet")
genetics = pl.read_parquet("Preprocessed_data/genetics.parquet")
center = pl.read_parquet("Preprocessed_data/center.parquet")
last_follow_up = pl.read_parquet("Preprocessed_data/last_follow_up.parquet")

#### Nettoyage de la base de données creat

On prend la toute première mesure de creat par patient tout en éliminant celles < 30.

In [ ]:
# =============================================================================
# SÉLECTION DE LA VISITE BASELINE
#
# Pour chaque patient, la visite baseline est définie comme la toute première
# mesure de DFGe disponible. Les patients dont cette première mesure présente
# un DFGe inférieur à 30 mL/min/1,73 m² sont exclus de l'étude.
# =============================================================================

creat_baseline = (
    creat
    # Trier les visites chronologiquement pour chaque patient
    .sort(["patient_id", "creat_date"])

    # Regrouper les observations par patient
    .group_by("patient_id")

    # Conserver la toute première visite de chaque patient
    .agg(pl.all().first())

    # Exclure les patients dont le DFGe de la première visite est < 30
    .filter(pl.col("ckd_epi") >= 30)
)

# Afficher les dimensions du jeu de données obtenu
print("La taille du dataset :", creat_baseline.shape)

# Vérifier qu'il reste une seule ligne par patient
print(
    "Nombre de patients :",
    creat_baseline.select(pl.col("patient_id").n_unique())
)

# inclusion.head()

#### Inclusion + creat_baseline

In [ ]:
# =============================================================================
# NETTOYAGE DES DONNÉES D'INCLUSION
# =============================================================================

# Conserver uniquement les patients dont le sexe est renseigné.
# Cette variable étant utilisée dans les analyses ultérieures, les patients
# présentant un sexe manquant sont exclus.
inclusion_clean = inclusion.filter(pl.col("sex").is_not_null())


# =============================================================================
# CONSTRUCTION DU JEU DE DONNÉES BASELINE
# =============================================================================

# Ajouter les caractéristiques démographiques et cliniques provenant des
# données d'inclusion au jeu de données baseline. La jointure est réalisée
# sur l'identifiant unique du patient. Une jointure "inner" est utilisée afin
# de conserver uniquement les patients présents dans creat_baseline et dont
# le sexe est renseigné.
df_baseline = creat_baseline.join(
    inclusion_clean.select([
        "patient_id",
        "sex",
        "inclusion_date",
        "age_inclusion",
        "birth_date",
        "has_diabetes",
        "has_hypertension",
        "age_hypertension",
        "has_hypertension_before_35",
        "diabetes_year",
        "has_high_cholesterol",
        "cholesterol_year",
        "has_urologic_event",
        "age_cysts_pain",
        "age_cysts_infection",
        "age_intra_cysts_hemorrhage",
        "age_lithiasis",
        "age_macro_hematuria",
        "has_urologic_event_before_35"
    ]),
    on="patient_id",
    how="inner"
)


# =============================================================================
# VÉRIFICATION DU JEU DE DONNÉES OBTENU
# =============================================================================

# Afficher les dimensions du jeu de données obtenu
print("La taille du dataset :", df_baseline.shape)

# Vérifier qu'il ne reste qu'une seule ligne par patient
print("Nombre de patients :", df_baseline.select(pl.col("patient_id").n_unique()))

# Vérifier qu'aucun patient ne présente un sexe manquant
print(
    "Nombre de sexes manquants :",
    df_baseline.select(pl.col("sex").is_null().sum())
)

# Calculer le pourcentage de valeurs manquantes pour chaque variable
df_baseline.null_count() / len(df_baseline) * 100

#### df_baseline + genetics

In [ ]:
# ==========================================================
# AJOUT DES DONNÉES GÉNÉTIQUES
# ==========================================================

# ----------------------------------------------------------
# Nettoyage et recodage des informations génétiques
# ----------------------------------------------------------

genetics_clean = (
    genetics

    # Exclure uniquement les patients ne disposant d'aucune information
    # génétique (gène et sous-groupe génétique absents). Les patients dont
    # le variant n'est pas renseigné sont conservés s'ils disposent d'une
    # information sur le gène ou le sous-groupe génétique.
    .filter(
        ~(
            pl.col("gene").is_null() &
            pl.col("gene_subgroup").is_null()
        )
    )

    .with_columns([

        # --------------------------------------------------
        # Détermination du type de variant
        # --------------------------------------------------
        # Les variants sont classés en trois catégories :
        # - truncating : variant tronquant
        # - missense   : variant non tronquant
        # - inconnu    : variant non renseigné

        pl.when(
            pl.col("variant").is_null() |
            (pl.col("variant").str.strip_chars() == "")
        )
        .then(pl.lit("inconnu"))

        .when(
            pl.col("variant")
            .str.to_lowercase()
            .is_in(["tru"])
        )
        .then(pl.lit("truncating"))

        .when(
            pl.col("variant").str.contains(
                r"(?i)del|dup|ins|fs|\*|splice|\+\d+|\-\d+"
            )
        )
        .then(pl.lit("truncating"))

        .otherwise(pl.lit("missense"))
        .alias("variant_type"),

        # --------------------------------------------------
        # Construction de la variable génétique utilisée
        # dans les modèles prédictifs
        # --------------------------------------------------
        # Les patients sont répartis dans cinq catégories :
        # - PKD1_truncating
        # - PKD1_non_truncating
        # - PKD2
        # - GUR : cas génétiques non résolus (status ≠ "solved")
        # - other : variant absent malgré une information génétique
        #           disponible ou gène différent de PKD1/PKD2.

        # Variant non renseigné
        pl.when(
            pl.col("variant").is_null() |
            (pl.col("variant").str.strip_chars() == "")
        )
        .then(pl.lit("other"))

        # Cas génétiques non résolus
        .when(pl.col("status") != "solved")
        .then(pl.lit("GUR"))

        # Mutation PKD2
        .when(pl.col("gene_subgroup") == "PKD2")
        .then(pl.lit("PKD2"))

        # Mutation PKD1 tronquante
        .when(
            (pl.col("gene_subgroup") == "PKD1") &
            (pl.col("is_truncating"))
        )
        .then(pl.lit("PKD1_truncating"))

        # Mutation PKD1 non tronquante
        .when(pl.col("gene_subgroup") == "PKD1")
        .then(pl.lit("PKD1_non_truncating"))

        # Tous les autres profils génétiques
        .otherwise(pl.lit("other"))
        .alias("genetic")

    ])
)


# ----------------------------------------------------------
# Ajout des informations génétiques au jeu de données baseline
# ----------------------------------------------------------

# Ajouter les informations génétiques aux patients du jeu de données baseline.
# Une jointure interne ("inner") est utilisée afin de conserver uniquement les
# patients disposant d'une catégorie génétique.
df_baseline_genetic = (
    df_baseline
    .join(
        genetics_clean.select([
            "patient_id",
            "variant",
            "status",
            "gene_subgroup",
            "variant_type",
            "genetic",
            "is_truncating"
        ]),
        on="patient_id",
        how="inner"
    )

    # Calcul de l'âge (en années) au moment de la mesure baseline
    .with_columns(
        (
            (pl.col("creat_date") - pl.col("birth_date"))
            .dt.total_days() / 365.25
        ).alias("creat_age")
    )
)


# ==========================================================
# VÉRIFICATION DU JEU DE DONNÉES
# ==========================================================

# Afficher les dimensions du jeu de données obtenu
print("La taille du dataset :", df_baseline_genetic.shape)

# Vérifier qu'il ne reste qu'une seule ligne par patient
print(
    "Nombre de patients :",
    df_baseline_genetic.select(pl.col("patient_id").n_unique())
)

# Calculer le pourcentage de valeurs manquantes pour chaque variable
# df_baseline_genetic.null_count() / len(df_baseline_genetic) * 100

#### df_baseline_genetic + last_follow_up

In [ ]:
# ==========================================================
# AJOUT DES DONNÉES DE SUIVI
# ==========================================================

# Ajouter les informations de suivi au jeu de données baseline.
# Une jointure gauche ("left") est utilisée afin de conserver tous
# les patients du jeu de données baseline.
df_survival = df_baseline_genetic.join(
    last_follow_up.select([
        "patient_id",
        "dialysis_date",
        "transplant_date",
        "kidney_failure_date",
        "last_follow_up_date",
        "is_alive",
        "death_date",
        "last_creat_date",
        "is_on_renal_replacement"
    ]),
    on="patient_id",
    how="left"
)


# ==========================================================
# SÉLECTION DES PATIENTS ÉLIGIBLES
# ==========================================================

# Exclure les patients dont le statut de traitement de suppléance
# rénale (RRT) n'est pas renseigné.

df_survival = df_survival.filter(
    pl.col("is_on_renal_replacement").is_not_null()
)


# ==========================================================
# DÉFINITION DE L'ÉVÉNEMENT D'INTÉRÊT
# ==========================================================

# Définir la date de l'événement comme la première date observée parmi :
# - la date d'insuffisance rénale terminale,
# - la date de début de dialyse,
# - la date de transplantation rénale.
df_survival = df_survival.with_columns(
    pl.min_horizontal([
        pl.col("kidney_failure_date"),
        pl.col("dialysis_date"),
        pl.col("transplant_date")
    ]).alias("event_date")
)


# Créer l'indicateur binaire de l'événement :
# 1 = IRT observée
# 0 = patient censuré.
df_survival = df_survival.with_columns(
    pl.col("event_date")
      .is_not_null()
      .cast(pl.Int8)
      .alias("event_irt")
)


# ==========================================================
# CONTRÔLE DE LA CHRONOLOGIE
# ==========================================================

# Exclure les patients dont l'événement est survenu avant la
# mesure baseline.

df_survival = df_survival.filter(
    ~(
        (pl.col("event_date").is_not_null()) &
        (pl.col("event_date") < pl.col("creat_date"))
    )
)


# ==========================================================
# CORRECTION DES DATES DE SUIVI
# ==========================================================

# Les dates de suivi postérieures au décès sont considérées comme
# incohérentes et remplacées par une valeur manquante.

df_survival = df_survival.with_columns([

    pl.when(
        (pl.col("death_date").is_not_null()) &
        (pl.col("last_follow_up_date") > pl.col("death_date"))
    )
    .then(None)
    .otherwise(pl.col("last_follow_up_date"))
    .alias("last_follow_up_date_clean"),

    pl.when(
        (pl.col("death_date").is_not_null()) &
        (pl.col("last_creat_date") > pl.col("death_date"))
    )
    .then(None)
    .otherwise(pl.col("last_creat_date"))
    .alias("last_creat_date_clean")
])


# ==========================================================
# DÉFINITION DE LA DATE DE FIN DE SUIVI
# ==========================================================

# La date de fin du suivi est définie selon la priorité suivante :
# 1. Date de l'événement d'IRT.
# 2. Date du décès.
# 3. Dernière date de suivi disponible.
#
# En l'absence d'événement et de décès, la date de suivi retenue
# correspond à la plus récente entre la dernière visite de suivi
# et la dernière mesure de créatinine disponibles.

df_survival = df_survival.with_columns(

    pl.when(pl.col("event_irt") == 1)
    .then(pl.col("event_date"))

    .otherwise(

        pl.when(pl.col("death_date").is_not_null())
        .then(pl.col("death_date"))

        .otherwise(
            pl.max_horizontal([
                pl.col("last_follow_up_date_clean"),
                pl.col("last_creat_date_clean")
            ])
        )
    )

    .alias("end_date")
)


# ==========================================================
# CALCUL DES TEMPS DE SURVIE
# ==========================================================

df_survival = df_survival.with_columns([

    # Temps écoulé entre la mesure baseline et la fin du suivi.
    ((pl.col("end_date") - pl.col("creat_date"))
     .dt.total_days() / 365.25)
    .alias("time_to_event"),

    # Âge du patient à la fin du suivi (événement ou censure).
    ((pl.col("end_date") - pl.col("birth_date"))
     .dt.total_days() / 365.25)
    .alias("age_end")

])


# ==========================================================
# VÉRIFICATION DU JEU DE DONNÉES
# ==========================================================

# Afficher les dimensions du jeu de données obtenu.
print("La taille du dataset :", df_survival.shape)

# Vérifier qu'il ne reste qu'une seule ligne par patient.
print(
    "Nombre de patients :",
    df_survival.select(pl.col("patient_id").n_unique())
)

# Calculer le pourcentage de valeurs manquantes pour chaque variable.
# df_survival.null_count() / len(df_survival) * 100

#### Nettoyage de la base de données

In [ ]:
# ==========================================================
# CONTRÔLE DE LA COHÉRENCE DES DATES
# ==========================================================

# Exclure les patients dont la date de fin de suivi est antérieure
# à la mesure baseline, ce qui conduirait à un temps de suivi négatif.
df_survival = df_survival.filter(
    pl.col("end_date") >= pl.col("creat_date")
)


# ==========================================================
# VÉRIFICATION DU JEU DE DONNÉES
# ==========================================================

# Afficher les dimensions du jeu de données obtenu.
print("La taille du dataset :", df_survival.shape)

# Vérifier qu'il ne reste qu'une seule ligne par patient.
print(
    "Nombre de patients :",
    df_survival.select(pl.col("patient_id").n_unique())
)

In [ ]:
# ==========================================================
# SÉLECTION DES PATIENTS ÉLIGIBLES
# ==========================================================

# Exclure les patients âgés de moins de 18 ans au moment
# de la mesure baseline.
df_survival = df_survival.filter(
    pl.col("creat_age") >= 18
)


# ==========================================================
# VÉRIFICATION DU JEU DE DONNÉES
# ==========================================================

# Afficher les dimensions du jeu de données obtenu.
print("La taille du dataset :", df_survival.shape)

# Vérifier qu'il ne reste qu'une seule ligne par patient.
print(
    "Nombre de patients :",
    df_survival.select(pl.col("patient_id").n_unique())
)

In [ ]:
# ==========================================================
# SÉLECTION DES PATIENTS ÉLIGIBLES
# ==========================================================

# Exclure les patients dont le temps de suivi est nul afin de
# conserver uniquement les patients présentant un suivi strictement positif.
df_survival_clean = df_survival.filter(
    pl.col("time_to_event") > 0
)


# ==========================================================
# VÉRIFICATION DU JEU DE DONNÉES
# ==========================================================

# Afficher les dimensions du jeu de données obtenu.
print("La taille du dataset :", df_survival_clean.shape)

# Vérifier qu'il ne reste qu'une seule ligne par patient.
print(
    "Nombre de patients :",
    df_survival_clean.select(pl.col("patient_id").n_unique())
)

In [ ]:
# ==========================================================
# CONSTRUCTION DE VARIABLES CLINIQUES DÉRIVÉES
# ==========================================================
#
# Construction de l'âge du premier événement urologique à
# partir des différentes complications urologiques.
#
# Cette variable est utilisée pour déterminer si un événement
# urologique est survenu avant la mesure baseline.
# ==========================================================

df_survival_clean = df_survival_clean.with_columns(

    pl.min_horizontal([
        "age_cysts_pain",
        "age_cysts_infection",
        "age_intra_cysts_hemorrhage",
        "age_lithiasis",
        "age_macro_hematuria"
    ]).alias("age_urologic_event")

)

In [ ]:
# ==========================================================
# CONSTRUCTION DES VARIABLES CLINIQUES À LA BASELINE
# ==========================================================
#
# Les variables hypertension et complications urologiques
# sont reconstruites à partir :
#   - de la présence de l'événement ;
#   - de l'âge de survenue lorsqu'il est connu ;
#   - de l'âge à la mesure baseline (creat_age).
#
# Codage retenu :
#   0 : absence d'événement avant la baseline
#       (absence de l'événement ou événement après la baseline)
#   1 : événement survenu avant ou à la baseline
#   2 : événement présent mais âge de survenue inconnu
#
# Ce codage permet de conserver les patients présentant
# l'événement lorsque l'information temporelle est absente.
# ==========================================================


# ==========================================================
# FONCTION UTILITAIRE
# ==========================================================
#
# Construit une variable clinique codée selon le statut de
# l'événement à la baseline.
# ==========================================================

def make_baseline_var(has_col, age_col, baseline_col):
    return (
        pl.when(pl.col(has_col) == False)
        .then(0)

        .when(
            (pl.col(has_col) == True) &
            (pl.col(age_col).is_not_null()) &
            (pl.col(age_col) <= pl.col(baseline_col))
        )
        .then(1)

        .when(
            (pl.col(has_col) == True) &
            (pl.col(age_col).is_null())
        )
        .then(2)

        .otherwise(0)
        .cast(pl.Int8)
    )


# ==========================================================
# CRÉATION DES VARIABLES CLINIQUES À LA BASELINE
# ==========================================================

df_survival_clean = df_survival_clean.with_columns([

    make_baseline_var(
        "has_hypertension",
        "age_hypertension",
        "creat_age"
    ).alias("has_hypertension_baseline"),

    make_baseline_var(
        "has_urologic_event",
        "age_urologic_event",
        "creat_age"
    ).alias("has_urologic_event_baseline"),

])

In [ ]:
# ==========================================================
# CONSTRUCTION DES VARIABLES CLINIQUES AVANT 35 ANS
# ==========================================================
#
# Les variables hypertension et complications urologiques
# sont reconstruites en fonction de leur survenue avant
# l'âge de 35 ans.
#
# Codage retenu :
#   0 : absence d'événement avant 35 ans
#       (absence de l'événement ou événement après 35 ans)
#   1 : événement survenu avant ou à 35 ans
#   2 : événement présent mais âge de survenue inconnu
#
# Ce codage permet de conserver les patients présentant
# l'événement lorsque l'information temporelle est absente.
# ==========================================================


# ==========================================================
# FONCTION UTILITAIRE
# ==========================================================
#
# Construit une variable clinique codée selon le statut de
# l'événement avant l'âge de 35 ans.
# ==========================================================

def before_35(has_col, age_col):
    return (
        pl.when(pl.col(has_col) == False)
        .then(0)

        .when(
            (pl.col(has_col) == True) &
            (pl.col(age_col).is_not_null()) &
            (pl.col(age_col) <= 35)
        )
        .then(1)

        .when(
            (pl.col(has_col) == True) &
            (pl.col(age_col).is_null())
        )
        .then(2)

        .otherwise(0)
        .cast(pl.Int8)
    )


# ==========================================================
# CRÉATION DES VARIABLES CLINIQUES AVANT 35 ANS
# ==========================================================

df_survival_clean = df_survival_clean.with_columns([

    before_35(
        "has_hypertension",
        "age_hypertension"
    ).alias("htn_before_35"),

    before_35(
        "has_urologic_event",
        "age_urologic_event"
    ).alias("uro_before_35")

])

#### Construire train.parquet et test.parquet

In [ ]:
# ==========================================================
# CRÉATION DES JEUX DE DÉVELOPPEMENT ET DE TEST EXTERNE
# ==========================================================

df_survival_final = df_survival_clean.with_columns(
    pl.col("patient_id").str.slice(0, 2).alias("center_id")
)

# external test set
external_test_df = df_survival_final.filter(
    pl.col("center_id").is_in(["09", "04"])
)

# development set
development_df  = df_survival_final.filter(
    ~pl.col("center_id").is_in(["09", "04"])
)

In [ ]:
# ==========================================================
# VÉRIFICATION DES JEUX DE DONNÉES
# ==========================================================

print(f"Dataset complet      : {df_survival_final.shape}")
print(f"Développement        : {development_df.shape}")
print(f"Test externe         : {external_test_df.shape}")

print("\nNombre de patients")
print(f"Complet              : {df_survival_final['patient_id'].n_unique()}")
print(f"Développement        : {development_df['patient_id'].n_unique()}")
print(f"Test externe         : {external_test_df['patient_id'].n_unique()}")

In [ ]:
# ==========================================================
# CRÉATION DU JEU D'ENTRAÎNEMENT ET DU TEST INTERNE
# ==========================================================
#
# Le jeu de développement est divisé en :
#   - jeu d'entraînement (80 %)
#   - jeu de test interne (20 %)
#
# La stratification est réalisée sur la variable d'événement
# afin de conserver la même proportion d'événements dans les
# deux ensembles.
# ==========================================================

train_df, internal_test_df = train_test_split(
    development_df.to_pandas(),
    test_size=0.20,
    random_state=42,
    stratify=development_df["event_irt"].to_pandas()
)

train_df = pl.from_pandas(train_df)
internal_test_df = pl.from_pandas(internal_test_df)

In [ ]:
# ==========================================================
# VÉRIFICATION DES JEUX DE DONNÉES
# ==========================================================

print(f"Dataset complet      : {df_survival_final.shape}")
print(f"Développement        : {development_df.shape}")
print(f"Entraînement         : {train_df.shape}")
print(f"Test interne         : {internal_test_df.shape}")
print(f"Test externe         : {external_test_df.shape}")

print("\nNombre de patients")
print(f"Complet              : {df_survival_final['patient_id'].n_unique()}")
print(f"Développement        : {development_df['patient_id'].n_unique()}")
print(f"Entraînement         : {train_df['patient_id'].n_unique()}")
print(f"Test interne         : {internal_test_df['patient_id'].n_unique()}")
print(f"Test externe         : {external_test_df['patient_id'].n_unique()}")

In [ ]:
# ==========================================================
# SAUVEGARDE DES JEUX DE DONNÉES
# ==========================================================

OUTPUT_DIR = Path("My_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_survival_final.write_parquet(
    OUTPUT_DIR / "df_survival_final.parquet"
)

development_df.write_parquet(
    OUTPUT_DIR / "development.parquet"
)

train_df.write_parquet(
    OUTPUT_DIR / "train.parquet"
)

internal_test_df.write_parquet(
    OUTPUT_DIR / "internal_test.parquet"
)

external_test_df.write_parquet(
    OUTPUT_DIR / "external_test.parquet"
)

print("Tous les jeux de données ont été sauvegardés avec succès.")

In [ ]:
# ==========================================================
# RÉPARTITION DES PATIENTS PAR CENTRE
# ==========================================================
#
# Cette analyse décrit la répartition des patients selon
# leur centre d'inclusion et résume, pour chaque centre :
#   - le nombre de patients ;
#   - le nombre d'événements d'IRT ;
#   - le nombre de patients censurés ;
#   - le pourcentage d'événements.
#
# Une ligne récapitulative est ajoutée afin d'obtenir les
# statistiques globales de la cohorte. Le tableau est ensuite
# exporté au format HTML pour être utilisé dans le rapport.
# ==========================================================


# ==========================================================
# AJOUT DES INFORMATIONS SUR LES CENTRES
# ==========================================================

df_survival_final = df_survival_final.join(
    center.select([
        "center_id",
        "center_name",
        "center_city"
    ]),
    on="center_id",
    how="left"
)


# ==========================================================
# CALCUL DES STATISTIQUES PAR CENTRE
# ==========================================================

df_center = (
    df_survival_final
    .group_by(["center_id", "center_name"])
    .agg([
        pl.len().alias("n_patients"),
        pl.col("event_irt").sum().alias("n_events"),
        (pl.col("event_irt") == 0).sum().alias("n_no_events"),
        (pl.col("event_irt").mean() * 100).alias("pct_events")
    ])
    .sort("n_patients", descending=True)
)


# ==========================================================
# AJOUT D'UNE LIGNE RÉCAPITULATIVE
# ==========================================================

df_total = (
    df_center
    .select([
        pl.lit("TOTAL").alias("center_id"),
        pl.lit("").alias("center_name"),
        pl.col("n_patients").sum().alias("n_patients"),
        pl.col("n_events").sum().alias("n_events"),
        pl.col("n_no_events").sum().alias("n_no_events"),
        (
            pl.col("n_events").sum()
            / pl.col("n_patients").sum() * 100
        ).alias("pct_events")
    ])
)

df_center_final = (
    pl.concat([df_center, df_total])
    .with_columns(
        pl.col("pct_events").round(1)
    )
)


# ==========================================================
# APERÇU DU TABLEAU
# ==========================================================

display(df_center_final)


# ==========================================================
# EXPORT DU TABLEAU AU FORMAT HTML
# ==========================================================

(
    df_center_final
    .to_pandas()
    .style
    .background_gradient(cmap="Blues")
    .to_html(
        r".\tables\machine_learning\summary_by_center.html"
    )
)

In [ ]:
# ==========================================================
# DESCRIPTION DU JEU DE DONNÉES FINAL
# ==========================================================
#
# Ce tableau résume les principales caractéristiques de la
# cohorte finale utilisée pour les analyses de survie.
# ==========================================================

summary_dataset = pl.DataFrame({
    "Caractéristique": [
        "Nombre de patients",
        "Événements IRT",
        "Patients censurés",
        "Taux d'événements (%)",
        "Âge baseline (min–max)",
        "Âge baseline (médiane)",
        "DFGe baseline (min–max)",
        "DFGe baseline (médiane)",
        "Suivi (années, médiane)",
        "Suivi (années, min–max)"
    ],
    "Valeur": [
        f"{df_survival_final.height}",
        f"{df_survival_final['event_irt'].sum()}",
        f"{(df_survival_final['event_irt'] == 0).sum()}",
        f"{df_survival_final['event_irt'].mean() * 100:.1f}",

        f"{df_survival_final['creat_age'].min():.1f}–{df_survival_final['creat_age'].max():.1f}",
        f"{df_survival_final['creat_age'].median():.1f}",

        f"{df_survival_final['ckd_epi'].min():.1f}–{df_survival_final['ckd_epi'].max():.1f}",
        f"{df_survival_final['ckd_epi'].median():.1f}",

        f"{df_survival_final['time_to_event'].median():.1f}",
        f"{df_survival_final['time_to_event'].min():.1f}–{df_survival_final['time_to_event'].max():.1f}"
    ]
})

summary_dataset

In [ ]:
# ==========================================================
# VÉRIFICATION DE LA RÉPARTITION DES ÉVÉNEMENTS
# ==========================================================
#
# Vérifie le nombre de patients, d'événements d'IRT et de
# patients censurés dans chaque jeu de données.
# ==========================================================

def check_events(df: pl.DataFrame, name: str) -> None:
    n = df.height
    n_events = int(df["event_irt"].sum())
    n_censored = n - n_events

    print(f"\n===== {name} =====")
    print(f"Patients       : {n}")
    print(f"Événements IRT : {n_events} ({100 * n_events / n:.1f}%)")
    print(f"Censurés       : {n_censored} ({100 * n_censored / n:.1f}%)")


check_events(df_survival_final, "Jeu complet")
check_events(development_df, "Développement")
check_events(train_df, "Entraînement")
check_events(internal_test_df, "Test interne")
check_events(external_test_df, "Test externe")